# 🧪 Scanntech QA — Validação Etapa 1

> **Escopo desta etapa:** Vendas simples, cancelamentos, desconto e acréscimo por operador, multi-pagamento, item pesável e fechamento por POS.
> Sem promoção Scanntech — apenas validação de estrutura e valores do roteiro.

### Como usar
Execute as células **na ordem** (▶️ de cima para baixo).

| Célula | O que faz |
|--------|-----------|
| 1 | Instala dependências |
| 2 | Clona o repositório |
| 3 | Upload da planilha do roteiro |
| 4 | Upload do export Audit (Excel) |
| 5 | Executa a validação |
| 6 | Exibe resumo e tabela de resultados |
| 7 | Baixa o arquivo Excel preenchido |


---
## 📦 Célula 1 — Instalação de dependências

In [ ]:
!pip install -q pandas openpyxl pdfplumber
print('✅ Dependências instaladas')

---
## 🗂️ Célula 2 — Clonar / atualizar repositório

In [ ]:
import os, sys

REPO_DIR = 'automacao_scann'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main --quiet
    print(f'✅ Repositório atualizado em: {os.getcwd()}')
else:
    !git clone https://github.com/Miked0/automacao_scann.git --quiet
    %cd {REPO_DIR}
    print(f'✅ Repositório clonado em: {os.getcwd()}')

# Adiciona src/ ao path para importar os módulos
SRC_PATH = os.path.join(os.getcwd(), 'src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f'📂 Módulos carregados de: {SRC_PATH}')

---
## 📋 Célula 3 — Upload da planilha do roteiro

> Faça upload do arquivo **TEMPLATE_COM_BIN_NOVO.xlsx** (ou equivalente).

In [ ]:
from google.colab import files as colab_files
import shutil, os

os.makedirs('input', exist_ok=True)
os.makedirs('output', exist_ok=True)

print('=' * 55)
print('📋 Faça upload da planilha do roteiro (.xlsx)')
print('=' * 55)

uploaded = colab_files.upload()

INPUT_PLANILHA = None
for fname in uploaded:
    dest = f'input/{fname}'
    shutil.copy(fname, dest)
    INPUT_PLANILHA = dest

if INPUT_PLANILHA:
    print(f'✅ Planilha carregada: {INPUT_PLANILHA}')
else:
    raise ValueError('❌ Nenhum arquivo enviado. Execute esta célula novamente.')

---
## 📊 Célula 4 — Upload do export Audit

> Faça upload do arquivo Excel exportado da aba **Audit** da API.
> Formato esperado: `export_tickets_audit_companyId-XXXXXX_*.xlsx`

In [ ]:
print('=' * 55)
print('📊 Faça upload do export Audit (.xlsx)')
print('=' * 55)

uploaded_audit = colab_files.upload()

INPUT_AUDIT = None
for fname in uploaded_audit:
    dest = f'input/{fname}'
    shutil.copy(fname, dest)
    INPUT_AUDIT = dest

if INPUT_AUDIT:
    import pandas as pd
    try:
        df_audit_preview = pd.read_excel(INPUT_AUDIT, nrows=3)
        print(f'✅ Audit carregado: {INPUT_AUDIT}')
        print(f'   Colunas detectadas: {list(df_audit_preview.columns[:8])}...')
    except Exception as e:
        print(f'⚠️ Arquivo carregado mas erro na leitura: {e}')
else:
    raise ValueError('❌ Nenhum arquivo enviado. Execute esta célula novamente.')

---
## ▶️ Célula 5 — Executar validação da Etapa 1

> Esta célula lê o roteiro, cruza com o Audit e executa todos os checks da Etapa 1.

In [ ]:
from pathlib import Path

# Importa módulos do projeto
from reader import load_roteiro_tests
from validators import validate_test_case
from exporters import export_results

OUTPUT_FILE = 'output/etapa1_resultado.xlsx'

print('🚀 Iniciando validação — Etapa 1')
print()

# 1. Lê os casos de teste da planilha
print('[1/4] Lendo roteiro...')
tests = load_roteiro_tests(Path(INPUT_PLANILHA))
print(f'      → {len(tests)} caso(s) encontrado(s)')

# 2. Lê o Audit
print('[2/4] Lendo Audit...')
import pandas as pd, json
df_audit = pd.read_excel(INPUT_AUDIT)
# Normaliza nomes de colunas
df_audit.columns = [str(c).strip() for c in df_audit.columns]
print(f'      → {len(df_audit)} linha(s) no Audit')

# 3. Monta índice de movimentos por número de cupom
print('[3/4] Indexando movimentos...')
audit_index = {}
COL_CUPOM_CANDIDATES = ['Ticket', 'NumeroCupom', 'Cupom', 'CupomFiscal', 'ticket', 'numero', 'Numero']
COL_CUPOM = next((c for c in COL_CUPOM_CANDIDATES if c in df_audit.columns), None)

if COL_CUPOM:
    for _, row in df_audit.iterrows():
        key = str(row[COL_CUPOM]).strip()
        if key not in audit_index:
            audit_index[key] = []
        audit_index[key].append(row.to_dict())
    print(f'      → {len(audit_index)} cupom(ns) indexado(s) pela coluna "{COL_CUPOM}"')
else:
    print(f'      ⚠️ Coluna de cupom não detectada automaticamente.')
    print(f'      Colunas disponíveis: {list(df_audit.columns)}')

# 4. Valida cada caso de teste
print('[4/4] Validando casos...')
results = []
for t in tests:
    result = validate_test_case(t)
    # Enriquece com info do Audit quando disponível
    num_cupom = str(t.get('num_cupom') or t.get('sat') or t.get('nfce') or '').strip()
    audit_rows = audit_index.get(num_cupom, [])
    result['audit_linhas_encontradas'] = len(audit_rows)
    result['audit_cupom_buscado'] = num_cupom
    if audit_rows:
        # Tenta extrair total do Audit para comparação extra
        COL_TOTAL_AUDIT = next((c for c in ['Total', 'ImporteTotal', 'total', 'importe'] if c in audit_rows[0]), None)
        if COL_TOTAL_AUDIT:
            from decimal import Decimal
            total_audit = Decimal(str(audit_rows[0].get(COL_TOTAL_AUDIT, 0) or 0))
            total_plan = Decimal(str(t.get('total', 0) or 0))
            diff = (total_audit - total_plan).copy_abs()
            result['audit_total'] = str(total_audit)
            result['audit_diff_total'] = str(diff)
            result['audit_total_ok'] = 'OK' if diff <= Decimal('0.05') else 'DIVERGENCIA'
            if result['audit_total_ok'] == 'DIVERGENCIA' and result['status_final'] == 'OK':
                result['status_final'] = 'ALERTA'
                result['alertas'] = (result.get('alertas') or '') + ' | Total diverge do Audit'
    else:
        result['audit_total'] = None
        result['audit_diff_total'] = None
        result['audit_total_ok'] = 'NAO_CRUZADO'
    results.append({**t, **result})

# 5. Exporta resultado
export_results(results, Path(OUTPUT_FILE))

ok = sum(1 for r in results if r.get('status_final') == 'OK')
alerta = sum(1 for r in results if r.get('status_final') == 'ALERTA')
erro = sum(1 for r in results if str(r.get('status_final', '')).startswith('ERRO'))
print()
print('✅ Validação concluída!')
print(f'   🟢 OK     : {ok}')
print(f'   🟡 ALERTA : {alerta}')
print(f'   🔴 ERRO   : {erro}')

---
## 📊 Célula 6 — Resumo e tabela de resultados

In [ ]:
import pandas as pd

df_result = pd.read_excel(OUTPUT_FILE, sheet_name='Resultado')

# ── Resumo por status ────────────────────────────────────────────────────────
print('=' * 55)
print('🎯 Etapa 1 — Resumo da Validação')
print('=' * 55)
print(f'Total de casos analisados : {len(df_result)}')
print()
print(df_result['status_final'].value_counts().rename('Qtd').to_string())
print()

# ── Casos com ERRO ou ALERTA para revisão ───────────────────────────────────
df_problemas = df_result[df_result['status_final'] != 'OK']
if len(df_problemas) > 0:
    print(f'⚠️ {len(df_problemas)} caso(s) requerem atenção:')
    print()
# ── Tabela completa ──────────────────────────────────────────────────────────
COLUNAS = ['bloco', 'teste', 'tipo_promo', 'pagamento_raw',
           'status_final', 'motivo_status', 'alertas',
           'audit_cupom_buscado', 'audit_linhas_encontradas',
           'audit_total', 'audit_diff_total', 'audit_total_ok']
COLUNAS = [c for c in COLUNAS if c in df_result.columns]

df_result[COLUNAS]

---
## 📥 Célula 7 — Download do resultado preenchido

In [ ]:
from google.colab import files as colab_files
colab_files.download(OUTPUT_FILE)
print(f'📥 Download iniciado: {OUTPUT_FILE}')

---
## 🔍 Célula 8 — Diagnóstico (opcional)

> Execute esta célula se quiser inspecionar casos específicos ou depurar o cruzamento com o Audit.

In [ ]:
# ✏️ Edite o número do teste que quer inspecionar
TESTE_INSPECIONAR = 1

# ─────────────────────────────────────────────────────────────────────────────
caso = next((t for t in tests if str(t.get('teste', '')).strip() == str(TESTE_INSPECIONAR)), None)

if caso is None:
    print(f'❌ Teste {TESTE_INSPECIONAR} não encontrado. Verifique o número.')
else:
    print(f'🔍 Inspecionando Teste {TESTE_INSPECIONAR}')
    print()
    print('── Dados do roteiro ──────────────────────────────')
    for k, v in caso.items():
        if v is not None and str(v).strip() not in ('', 'nan'):
            print(f'  {k:25s}: {v}')
    print()
    num_cupom = str(caso.get('num_cupom') or caso.get('sat') or caso.get('nfce') or '').strip()
    print(f'── Linhas do Audit para cupom "{num_cupom}" ──────────')
    rows_audit = audit_index.get(num_cupom, [])
    if rows_audit:
        for i, row in enumerate(rows_audit):
            print(f'  Linha {i+1}: {dict(list(row.items())[:10])}')
    else:
        print(f'  ⚠️ Nenhuma linha encontrada no Audit para este cupom')
        if audit_index:
            print(f'  Chaves disponíveis no Audit (primeiras 10): {list(audit_index.keys())[:10]}')